In [ ]:
from pathlib import Path
from datetime import datetime, date, timedelta
import itertools
import math
import os
import re
import time
import warnings

import numpy as np
import pandas as pd
import openpyxl.reader.excel as openpyxl_excel

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    import seaborn as sns
except Exception:
    sns = None

from scipy import sparse
from scipy.special import expit
from scipy.optimize import minimize
from scipy.stats import chi2_contingency, norm

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression, PoissonRegressor
from sklearn.manifold import TSNE
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.decomposition import LatentDirichletAllocation

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

_OPENPYXL_ORIGINAL_READ_PROPERTIES = openpyxl_excel.ExcelReader.read_properties

def _openpyxl_read_properties_lenient(self):
    try:
        return _OPENPYXL_ORIGINAL_READ_PROPERTIES(self)
    except Exception as exc:

        print(f"Warning: skipped malformed workbook metadata: {exc}")
        return None

openpyxl_excel.ExcelReader.read_properties = _openpyxl_read_properties_lenient

RANDOM_STATE = 42
PROJECT_DIR = Path.cwd()
DOWNLOADS_DIR = Path.home() / "Downloads"
DATA_DIR_CANDIDATES = [
    Path(r"C:\Users\rendu\Downloads\NEISS Data\NEISS Data"),
    PROJECT_DIR / "data" / "neiss",
    PROJECT_DIR / "data",
    PROJECT_DIR,
    DOWNLOADS_DIR,
]
OUTPUT_DIR = PROJECT_DIR / "outputs" / "neiss_world_cup_full"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
MODEL_DIR = OUTPUT_DIR / "models"
for d in [OUTPUT_DIR, TABLE_DIR, FIGURE_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

REQUESTED_YEARS = list(range(1999, 2027))
PRIMARY_WC_FLAG = "WorldCup_year_any"

print(f"Project directory: {PROJECT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")


Project directory: D:\ai-job-agent\MIMIC_Project_3
Output directory: D:\ai-job-agent\MIMIC_Project_3\outputs\neiss_world_cup_full



[executed in 4.6s]


In [ ]:
WORLD_CUP_EVENTS = pd.DataFrame([
    {"event": "FIFA Women's World Cup", "event_type": "Women", "year": 1999, "start": "1999-06-19", "end": "1999-07-10"},
    {"event": "FIFA Men's World Cup", "event_type": "Men", "year": 2002, "start": "2002-05-31", "end": "2002-06-30"},
    {"event": "FIFA Women's World Cup", "event_type": "Women", "year": 2003, "start": "2003-09-20", "end": "2003-10-12"},
    {"event": "FIFA Men's World Cup", "event_type": "Men", "year": 2006, "start": "2006-06-09", "end": "2006-07-09"},
    {"event": "FIFA Women's World Cup", "event_type": "Women", "year": 2007, "start": "2007-09-10", "end": "2007-09-30"},
    {"event": "FIFA Men's World Cup", "event_type": "Men", "year": 2010, "start": "2010-06-11", "end": "2010-07-11"},
    {"event": "FIFA Women's World Cup", "event_type": "Women", "year": 2011, "start": "2011-06-26", "end": "2011-07-17"},
    {"event": "FIFA Men's World Cup", "event_type": "Men", "year": 2014, "start": "2014-06-12", "end": "2014-07-13"},
    {"event": "FIFA Women's World Cup", "event_type": "Women", "year": 2015, "start": "2015-06-06", "end": "2015-07-05"},
    {"event": "FIFA Men's World Cup", "event_type": "Men", "year": 2018, "start": "2018-06-14", "end": "2018-07-15"},
    {"event": "FIFA Women's World Cup", "event_type": "Women", "year": 2019, "start": "2019-06-07", "end": "2019-07-07"},
    {"event": "FIFA Men's World Cup", "event_type": "Men", "year": 2022, "start": "2022-11-20", "end": "2022-12-18"},
    {"event": "FIFA Women's World Cup", "event_type": "Women", "year": 2023, "start": "2023-07-20", "end": "2023-08-20"},
    {"event": "FIFA Men's World Cup", "event_type": "Men", "year": 2026, "start": "2026-06-11", "end": "2026-07-19"},
])
WORLD_CUP_EVENTS["start"] = pd.to_datetime(WORLD_CUP_EVENTS["start"])
WORLD_CUP_EVENTS["end"] = pd.to_datetime(WORLD_CUP_EVENTS["end"])

WORLD_CUP_EVENTS.to_csv(TABLE_DIR / "world_cup_event_calendar.csv", index=False)
WORLD_CUP_EVENTS



[executed in 0.0s]


In [ ]:
def find_neiss_files():
    files = {}
    for base in DATA_DIR_CANDIDATES:
        if not base.exists():
            continue
        for path in sorted(base.rglob("neiss*.xlsx")):
            m = re.search(r"(19|20)\d{2}", path.name)
            if not m:
                continue
            year = int(m.group(0))
            files.setdefault(year, path)
    return dict(sorted(files.items()))

neiss_files = find_neiss_files()

availability_rows = []
for year in REQUESTED_YEARS:
    p = neiss_files.get(year)
    availability_rows.append({
        "year": year,
        "file_found": bool(p),
        "path": str(p) if p else "",
        "size_mb": round(p.stat().st_size / (1024**2), 1) if p else np.nan,
    })
availability = pd.DataFrame(availability_rows)
availability["world_cup_year_any"] = availability["year"].isin(WORLD_CUP_EVENTS["year"])
availability.to_csv(TABLE_DIR / "data_availability_audit.csv", index=False)

print("NEISS files found:")
print(availability[availability["file_found"]].to_string(index=False))
print("\nMissing requested years:")
print(availability.loc[~availability["file_found"], "year"].tolist())

if not neiss_files:
    raise FileNotFoundError(
        "No annual NEISS Excel files were found. Place files named neissYYYY.xlsx in "
        f"{PROJECT_DIR / 'data' / 'neiss'} or in Downloads, then rerun."
    )


NEISS files found:
 year  file_found                                                          path  size_mb  world_cup_year_any
 1999        True C:\Users\rendu\Downloads\NEISS Data\NEISS Data\neiss1999.xlsx     28.7                True
 2000        True C:\Users\rendu\Downloads\NEISS Data\NEISS Data\neiss2000.xlsx     31.1               False
 2001        True C:\Users\rendu\Downloads\NEISS Data\NEISS Data\neiss2001.xlsx     32.8               False
 2002        True C:\Users\rendu\Downloads\NEISS Data\NEISS Data\neiss2002.xlsx     33.4                True
 2003        True C:\Users\rendu\Downloads\NEISS Data\NEISS Data\neiss2003.xlsx     32.2                True
 2004        True C:\Users\rendu\Downloads\NEISS Data\NEISS Data\neiss2004.xlsx     32.8               False
 2005        True C:\Users\rendu\Downloads\NEISS Data\NEISS Data\neiss2005.xlsx     33.3               False
 2006        True C:\Users\rendu\Downloads\NEISS Data\NEISS Data\neiss2006.xlsx     33.5                True



[executed in 0.1s]


In [ ]:
def norm_col(x):
    return re.sub(r"[^a-z0-9]+", "", str(x).strip().lower())

CANONICAL_ALIASES = {
    "case_id": ["cpsccasenumber", "cpsccase", "casenumber", "caseid"],
    "Treatment_Date": ["treatmentdate", "trmtdate", "dateoftreatment", "treatdate"],
    "Age": ["age"],
    "Sex": ["sex", "gender"],
    "Race": ["race"],
    "Other_Race": ["otherrace"],
    "Hispanic": ["hispanic", "ethnicity"],
    "Body_Part": ["bodypart", "bdypart"],
    "Diagnosis": ["diagnosis", "diag"],
    "Other_Diagnosis": ["otherdiagnosis", "othdiag"],
    "Body_Part_2": ["bodypart2", "bdypart2"],
    "Diagnosis_2": ["diagnosis2", "diag2"],
    "Other_Diagnosis_2": ["otherdiagnosis2", "othdiag2"],
    "Disposition": ["disposition", "disp"],
    "Location": ["location", "loc"],
    "Fire_Involvement": ["fireinvolvement", "fire"],
    "Product_1": ["product1", "prod1"],
    "Product_2": ["product2", "prod2"],
    "Product_3": ["product3", "prod3"],
    "Alcohol": ["alcohol", "alcoholinvolved", "alc"],
    "Drug": ["drug", "druginvolved"],
    "Narrative_1": ["narrative1", "narrative", "narr1"],
    "Narrative_2": ["narrative2", "narr2"],
    "Stratum": ["stratum"],
    "PSU": ["psu"],
    "Weight": ["weight", "wt"],
}

KEEP_COLS = list(CANONICAL_ALIASES.keys())

def choose_data_sheet(path):
    xl = pd.ExcelFile(path, engine="openpyxl")
    preferred = [s for s in xl.sheet_names if s.upper().startswith("NEISS_") and "FMT" not in s.upper()]
    if preferred:
        return preferred[0]
    non_fmt = [s for s in xl.sheet_names if "FMT" not in s.upper()]
    return non_fmt[0]

def read_format_sheet(path, year):
    try:
        xl = pd.ExcelFile(path, engine="openpyxl")
        fmt_sheets = [s for s in xl.sheet_names if "FMT" in s.upper()]
        if not fmt_sheets:
            return pd.DataFrame()
        fmt = pd.read_excel(path, sheet_name=fmt_sheets[0], engine="openpyxl")
        fmt["source_year"] = year
        return fmt
    except Exception as e:
        print(f"Could not read format sheet for {path.name}: {e}")
        return pd.DataFrame()

def standardize_year_file(path, year):
    sheet = choose_data_sheet(path)
    header = pd.read_excel(path, sheet_name=sheet, nrows=0, engine="openpyxl")
    norm_to_real = {norm_col(c): c for c in header.columns}
    rename = {}
    usecols = []
    for canonical, aliases in CANONICAL_ALIASES.items():
        found = None
        for alias in aliases:
            if alias in norm_to_real:
                found = norm_to_real[alias]
                break
        if found is not None:
            rename[found] = canonical
            usecols.append(found)
    df_year = pd.read_excel(path, sheet_name=sheet, usecols=usecols, engine="openpyxl")
    df_year = df_year.rename(columns=rename)
    for col in KEEP_COLS:
        if col not in df_year.columns:
            df_year[col] = pd.NA
    df_year = df_year[KEEP_COLS].copy()
    df_year["source_year_from_filename"] = year
    df_year["source_file"] = str(path)
    df_year["source_sheet"] = sheet
    return df_year

frames = []
fmt_frames = []
load_log = []
for year, path in neiss_files.items():
    t0 = time.time()
    try:
        part = standardize_year_file(path, year)
        frames.append(part)
        fmt = read_format_sheet(path, year)
        if len(fmt):
            fmt_frames.append(fmt)
        load_log.append({"year": year, "file": str(path), "rows": len(part), "status": "loaded", "seconds": round(time.time() - t0, 1)})
        print(f"Loaded {year}: {len(part):,} rows from {path.name}")
    except Exception as e:
        load_log.append({"year": year, "file": str(path), "rows": 0, "status": f"failed: {e}", "seconds": round(time.time() - t0, 1)})
        print(f"FAILED {year}: {e}")

load_log = pd.DataFrame(load_log)
load_log.to_csv(TABLE_DIR / "load_log.csv", index=False)

df = pd.concat(frames, ignore_index=True)
fmt_all = pd.concat(fmt_frames, ignore_index=True) if fmt_frames else pd.DataFrame()

print(f"\nCombined rows: {len(df):,}")
print(load_log.to_string(index=False))


Loaded 1999: 312,943 rows from neiss1999.xlsx
Loaded 2000: 339,244 rows from neiss2000.xlsx
Loaded 2001: 356,038 rows from neiss2001.xlsx
Loaded 2002: 359,980 rows from neiss2002.xlsx
Loaded 2003: 347,380 rows from neiss2003.xlsx
Loaded 2004: 353,394 rows from neiss2004.xlsx
Loaded 2005: 360,374 rows from neiss2005.xlsx
Loaded 2006: 363,616 rows from neiss2006.xlsx
Loaded 2007: 369,841 rows from neiss2007.xlsx
Loaded 2008: 374,260 rows from neiss2008.xlsx
Loaded 2009: 391,944 rows from neiss2009.xlsx
Loaded 2010: 405,710 rows from neiss2010.xlsx
Loaded 2011: 396,502 rows from neiss2011.xlsx
Loaded 2012: 394,383 rows from neiss2012.xlsx
Loaded 2013: 376,926 rows from neiss2013.xlsx
Loaded 2014: 367,493 rows from neiss2014.xlsx
Loaded 2015: 359,129 rows from neiss2015.xlsx
Loaded 2016: 375,197 rows from neiss2016.xlsx
Loaded 2017: 386,906 rows from neiss2017.xlsx
Loaded 2018: 361,667 rows from neiss2018.xlsx
Loaded 2019: 358,715 rows from neiss2019.xlsx
Loaded 2020: 309,370 rows from nei


[executed in 2630.6s]


In [ ]:
def build_label_maps(fmt):
    maps = {}
    if fmt.empty:
        return maps
    cols = {norm_col(c): c for c in fmt.columns}
    name_col = cols.get("formatname")
    start_col = cols.get("startingvalueforformat")
    label_col = cols.get("formatvaluelabel")
    if not all([name_col, start_col, label_col]):
        return maps
    tmp = fmt[[name_col, start_col, label_col]].dropna(subset=[name_col, start_col, label_col]).copy()
    for fmt_name, g in tmp.groupby(name_col):
        mapping = {}
        for _, row in g.iterrows():
            try:
                code_value = int(row[start_col])
            except Exception:
                continue
            label = str(row[label_col]).strip()
            label = re.sub(r"^\s*\d+\s*-\s*", "", label).strip()
            mapping[code_value] = label
        maps[str(fmt_name).upper()] = mapping
    return maps

label_maps = build_label_maps(fmt_all)
print("Label maps available:", sorted(label_maps.keys()))

def map_label(series, fmt_name, unknown="Unknown"):
    mapping = label_maps.get(fmt_name, {})
    return pd.to_numeric(series, errors="coerce").map(mapping).fillna(unknown)

def neiss_age_to_years(age):
    x = pd.to_numeric(age, errors="coerce")
    out = x.astype("float64")
    months_mask = (x >= 201) & (x <= 223)
    out.loc[months_mask] = (x.loc[months_mask] - 200) / 12.0
    out.loc[(out < 0) | (out > 120)] = np.nan
    return out

df["Treatment_Date"] = pd.to_datetime(df["Treatment_Date"], errors="coerce")
df["Year"] = df["Treatment_Date"].dt.year.fillna(df["source_year_from_filename"]).astype("Int64")
df["Age_years"] = neiss_age_to_years(df["Age"])
df["Sex_label"] = map_label(df["Sex"], "SEX")
df["Race_label"] = map_label(df["Race"], "RACE")
df["Body_Part_label"] = map_label(df["Body_Part"], "BDYPT")
df["Diagnosis_label"] = map_label(df["Diagnosis"], "DIAG")
df["Disposition_label"] = map_label(df["Disposition"], "DISP")
df["Location_label"] = map_label(df["Location"], "LOC")
df["Weight"] = pd.to_numeric(df["Weight"], errors="coerce").fillna(1.0)

for col in ["Product_1", "Product_2", "Product_3", "Sex", "Race", "Body_Part", "Diagnosis", "Disposition", "Location"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in ["Narrative_1", "Narrative_2", "Other_Diagnosis", "Other_Diagnosis_2"]:
    df[col] = df[col].fillna("").astype(str)

df["Narrative_clean"] = (
    df[["Narrative_1", "Narrative_2", "Other_Diagnosis", "Other_Diagnosis_2"]]
    .agg(" ".join, axis=1)
    .str.lower()
    .str.replace(r"[^a-z0-9\s\-']", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

valid_cleaning_audit = pd.DataFrame({
    "metric": [
        "total_records",
        "missing_treatment_date",
        "invalid_or_missing_age",
        "missing_or_unknown_sex",
        "missing_weight",
    ],
    "n": [
        len(df),
        int(df["Treatment_Date"].isna().sum()),
        int(df["Age_years"].isna().sum()),
        int((~df["Sex"].isin([1, 2])).sum()),
        int(df["Weight"].isna().sum()),
    ],
})
valid_cleaning_audit.to_csv(TABLE_DIR / "cleaning_audit.csv", index=False)

print(valid_cleaning_audit.to_string(index=False))
print("\nDate range:", df["Treatment_Date"].min(), "to", df["Treatment_Date"].max())
print("Years in loaded data:", sorted([int(y) for y in df["Year"].dropna().unique()]))


Label maps available: ['AGELTTWO', 'ALC_DRUG', 'BDYPT', 'DIAG', 'DISP', 'FIRE', 'GENDER', 'HISP', 'LOC', 'PROD', 'RACE', 'SEX']
                metric       n
         total_records 9794932
missing_treatment_date       0
invalid_or_missing_age       0
missing_or_unknown_sex    1571
        missing_weight       0

Date range: 1999-01-01 00:00:00 to 2025-12-31 00:00:00
Years in loaded data: [1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]



[executed in 185.4s]


In [ ]:
def season_from_month(m):
    if m in [12, 1, 2]:
        return "Winter"
    if m in [3, 4, 5]:
        return "Spring"
    if m in [6, 7, 8]:
        return "Summer"
    if m in [9, 10, 11]:
        return "Fall"
    return "Unknown"

def fourth_thursday(year, month=11):
    d = date(year, month, 1)
    thursdays = []
    while d.month == month:
        if d.weekday() == 3:
            thursdays.append(d)
        d += timedelta(days=1)
    return thursdays[3]

def is_us_holiday(ts):
    if pd.isna(ts):
        return False
    d = ts.date()
    fixed = {(1, 1), (7, 4), (12, 24), (12, 25), (12, 31)}
    if (d.month, d.day) in fixed:
        return True
    try:
        if d == fourth_thursday(d.year, 11):
            return True
    except Exception:
        pass
    return False

df["month"] = df["Treatment_Date"].dt.month
df["month_name"] = df["Treatment_Date"].dt.month_name()
df["week_of_year"] = df["Treatment_Date"].dt.isocalendar().week.astype("Int64")
df["day_of_week"] = df["Treatment_Date"].dt.dayofweek
df["day_of_week_name"] = df["Treatment_Date"].dt.day_name()
df["week_start"] = df["Treatment_Date"] - pd.to_timedelta(df["Treatment_Date"].dt.dayofweek, unit="D")
df["season"] = df["month"].map(season_from_month)
fixed_holidays = {"01-01", "07-04", "12-24", "12-25", "12-31"}
month_day = df["Treatment_Date"].dt.strftime("%m-%d")
years_for_holidays = [int(y) for y in df["Year"].dropna().unique()]
thanksgiving_dates = {fourth_thursday(y, 11) for y in years_for_holidays}
df["holiday_flag"] = month_day.isin(fixed_holidays) | df["Treatment_Date"].dt.date.isin(thanksgiving_dates)

mens_years = set(WORLD_CUP_EVENTS.loc[WORLD_CUP_EVENTS["event_type"] == "Men", "year"])
womens_years = set(WORLD_CUP_EVENTS.loc[WORLD_CUP_EVENTS["event_type"] == "Women", "year"])
any_years = set(WORLD_CUP_EVENTS["year"])

df["WorldCup_year_men"] = df["Year"].astype("float").isin(mens_years)
df["WorldCup_year_women"] = df["Year"].astype("float").isin(womens_years)
df["WorldCup_year_any"] = df["Year"].astype("float").isin(any_years)

df["WorldCup_period_any"] = False
df["WorldCup_event_name"] = ""
df["WorldCup_period_any_plus14"] = False
for _, ev in WORLD_CUP_EVENTS.iterrows():
    mask = df["Treatment_Date"].between(ev["start"], ev["end"], inclusive="both")
    df.loc[mask, "WorldCup_period_any"] = True
    df.loc[mask, "WorldCup_event_name"] = f"{ev['event']} {ev['year']}"
    buffer_mask = df["Treatment_Date"].between(
        ev["start"] - pd.Timedelta(days=14),
        ev["end"] + pd.Timedelta(days=14),
        inclusive="both",
    )
    df.loc[buffer_mask, "WorldCup_period_any_plus14"] = True

df["same_calendar_wc_window"] = False
date_year = df["Treatment_Date"].dt.year.astype("Int64")
for _, ev in WORLD_CUP_EVENTS.iterrows():
    start_mmdd = ev["start"].strftime("-%m-%d")
    end_mmdd = ev["end"].strftime("-%m-%d")
    start_same_year = pd.to_datetime(date_year.astype(str) + start_mmdd, errors="coerce")
    end_same_year = pd.to_datetime(date_year.astype(str) + end_mmdd, errors="coerce")
    mask = (df["Treatment_Date"] >= start_same_year) & (df["Treatment_Date"] <= end_same_year)
    df.loc[mask, "same_calendar_wc_window"] = True

time_audit = df.groupby("Year", dropna=False).agg(
    records=("case_id", "size"),
    weighted_estimate=("Weight", "sum"),
    world_cup_year_any=("WorldCup_year_any", "max"),
    in_actual_wc_period=("WorldCup_period_any", "sum"),
).reset_index()
time_audit.to_csv(TABLE_DIR / "time_variable_audit_by_year.csv", index=False)
time_audit



[executed in 268.9s]


In [ ]:
prod_map = label_maps.get("PROD", {})
soccer_product_codes = sorted([code for code, label in prod_map.items() if "SOCCER" in str(label).upper()])
football_product_codes = sorted([code for code, label in prod_map.items() if "FOOTBALL" in str(label).upper()])
tv_product_codes = sorted([code for code, label in prod_map.items() if "TELEVISION" in str(label).upper() or str(label).upper().startswith("TV")])

SOCCER_TEXT_RE = re.compile(
    r"\b(soccer|futbol|fifa|world cup|goalie|goalkeeper|goal keeper|penalty kick|corner kick|soccer ball|shin guard|cleat)\b",
    re.IGNORECASE,
)
FOOTBALL_SOCCER_CONTEXT_RE = re.compile(
    r"\bfootball\b.{0,40}\b(match|goal|world cup|fifa|soccer|fan|watch|tv|television)\b|\b(match|goal|world cup|fifa|soccer|fan|watch|tv|television)\b.{0,40}\bfootball\b",
    re.IGNORECASE,
)
WATCHING_RE = re.compile(r"\b(watch|watching|watched|tv|television|screen|game on|match on|fan|viewing)\b", re.IGNORECASE)
PLAYING_RE = re.compile(r"\b(play|playing|played|practice|practicing|kicked|kick|running|field|goalie|goalkeeper|cleat|ball)\b", re.IGNORECASE)
CELEBRATION_RE = re.compile(r"\b(celebrat|cheer|jumped|party|bar|pub|gathering|tailgate|crowd|fight|altercation)\b", re.IGNORECASE)

product_cols = ["Product_1", "Product_2", "Product_3"]
soccer_code_set = set(soccer_product_codes)
tv_code_set = set(tv_product_codes)
df["soccer_product_flag"] = df[product_cols].isin(soccer_code_set).any(axis=1)
df["football_product_flag"] = df[product_cols].isin(set(football_product_codes)).any(axis=1)
df["tv_product_flag"] = df[product_cols].isin(tv_code_set).any(axis=1)
df["soccer_keyword_flag"] = df["Narrative_clean"].str.contains(SOCCER_TEXT_RE, na=False) | df["Narrative_clean"].str.contains(FOOTBALL_SOCCER_CONTEXT_RE, na=False)
df["watching_context_flag"] = df["Narrative_clean"].str.contains(WATCHING_RE, na=False) | df["tv_product_flag"]
df["playing_context_flag"] = df["Narrative_clean"].str.contains(PLAYING_RE, na=False) | df["soccer_product_flag"]
df["celebration_context_flag"] = df["Narrative_clean"].str.contains(CELEBRATION_RE, na=False)
df["soccer_related"] = df["soccer_product_flag"] | df["soccer_keyword_flag"]

def context_category(row):
    if not row["soccer_related"]:
        return "Non-soccer or unclear"
    if row["watching_context_flag"]:
        return "Watching or TV context"
    if row["celebration_context_flag"]:
        return "Celebration/gathering context"
    if row["playing_context_flag"]:
        return "Playing soccer context"
    return "Soccer context unclear"

df["soccer_context_category"] = df[["soccer_related", "watching_context_flag", "celebration_context_flag", "playing_context_flag"]].apply(context_category, axis=1)

keyword_audit = pd.DataFrame({
    "definition": [
        "soccer_product_flag",
        "soccer_keyword_flag",
        "soccer_related_combined",
        "watching_context_flag",
        "playing_context_flag",
        "celebration_context_flag",
    ],
    "records": [
        int(df["soccer_product_flag"].sum()),
        int(df["soccer_keyword_flag"].sum()),
        int(df["soccer_related"].sum()),
        int(df["watching_context_flag"].sum()),
        int(df["playing_context_flag"].sum()),
        int(df["celebration_context_flag"].sum()),
    ],
    "weighted_estimate": [
        df.loc[df["soccer_product_flag"], "Weight"].sum(),
        df.loc[df["soccer_keyword_flag"], "Weight"].sum(),
        df.loc[df["soccer_related"], "Weight"].sum(),
        df.loc[df["watching_context_flag"], "Weight"].sum(),
        df.loc[df["playing_context_flag"], "Weight"].sum(),
        df.loc[df["celebration_context_flag"], "Weight"].sum(),
    ],
})
keyword_audit.to_csv(TABLE_DIR / "soccer_definition_audit.csv", index=False)
print("Soccer product codes detected from NEISS_FMT:", soccer_product_codes)
keyword_audit


Soccer product codes detected from NEISS_FMT: [1267, 3225, 3241, 3271]



[executed in 440.1s]


In [ ]:
def weighted_mean_sd(x, w):
    mask = x.notna() & w.notna()
    if mask.sum() == 0:
        return np.nan, np.nan
    xv = x.loc[mask].astype(float)
    wv = w.loc[mask].astype(float)
    if wv.sum() <= 0:
        return np.nan, np.nan
    mean = np.average(xv, weights=wv)
    var = np.average((xv - mean) ** 2, weights=wv)
    return mean, math.sqrt(var)

def pct_fmt(n, denom):
    if denom == 0 or pd.isna(denom):
        return np.nan
    return 100 * n / denom

def fdr_bh(p_values):
    p = np.asarray([np.nan if v is None else v for v in p_values], dtype=float)
    out = np.full_like(p, np.nan, dtype=float)
    mask = ~np.isnan(p)
    if mask.sum() == 0:
        return out
    vals = p[mask]
    order = np.argsort(vals)
    ranked = vals[order]
    n = len(ranked)
    adj = ranked * n / (np.arange(n) + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    adj = np.clip(adj, 0, 1)
    tmp = np.empty(n)
    tmp[order] = adj
    out[mask] = tmp
    return out

def weighted_table_by_group(data, group_col, var_col, label_col=None, top_n=None):
    d = data.copy()
    label = label_col or var_col
    if label not in d.columns:
        d[label] = d[var_col].astype(str)
    if top_n:
        top_levels = d.groupby(label)["Weight"].sum().sort_values(ascending=False).head(top_n).index
        d = d[d[label].isin(top_levels)].copy()
    groups = sorted(d[group_col].dropna().unique().tolist())
    rows = []
    for level, g_level in d.groupby(label, dropna=False):
        total_level_n = len(g_level)
        total_level_w = g_level["Weight"].sum()
        p_for_level = np.nan
        if len(groups) >= 2:
            try:
                ct = pd.crosstab(d[group_col], d[label] == level)
                if ct.shape[0] >= 2 and ct.shape[1] == 2:
                    p_for_level = chi2_contingency(ct)[1]
            except Exception:
                p_for_level = np.nan
        for group_value in groups:
            g = d[d[group_col] == group_value]
            sub = g[g[label] == level]
            rows.append({
                "group": group_value,
                "variable": var_col,
                "level": level,
                "unweighted_n": len(sub),
                "weighted_estimate": sub["Weight"].sum(),
                "weighted_pct_within_group": pct_fmt(sub["Weight"].sum(), g["Weight"].sum()),
                "p_value_unweighted_chisq": p_for_level,
                "overall_unweighted_n": total_level_n,
                "overall_weighted_estimate": total_level_w,
            })
    out = pd.DataFrame(rows)
    if "p_value_unweighted_chisq" in out:
        out["fdr_q_value"] = fdr_bh(out["p_value_unweighted_chisq"].values)
    return out

def save_table(df_table, name):
    csv_path = TABLE_DIR / f"{name}.csv"
    xlsx_path = TABLE_DIR / f"{name}.xlsx"
    df_table.to_csv(csv_path, index=False)
    try:
        df_table.to_excel(xlsx_path, index=False)
    except Exception as e:
        print(f"Excel export skipped for {name}: {e}")
    print(f"Saved table: {csv_path}")
    return csv_path

def save_current_fig(name):
    path = FIGURE_DIR / f"{name}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved figure: {path}")
    return path



[executed in 0.0s]


In [ ]:

df["age_group"] = pd.cut(
    df["Age_years"],
    bins=[-0.001, 4, 14, 24, 44, 64, 120],
    labels=["0-4", "5-14", "15-24", "25-44", "45-64", "65+"],
)

table1_rows = []
for group_value, g in df.groupby(PRIMARY_WC_FLAG, dropna=False):
    mean_age, sd_age = weighted_mean_sd(g["Age_years"], g["Weight"])
    table1_rows.append({
        "group": group_value,
        "variable": "Overall",
        "level": "All records",
        "unweighted_n": len(g),
        "weighted_estimate": g["Weight"].sum(),
        "weighted_pct_within_group": 100.0,
        "age_weighted_mean": mean_age,
        "age_weighted_sd": sd_age,
    })

table1_parts = [pd.DataFrame(table1_rows)]
for var, label in [
    ("Sex", "Sex_label"),
    ("Race", "Race_label"),
    ("age_group", "age_group"),
]:
    table1_parts.append(weighted_table_by_group(df, PRIMARY_WC_FLAG, var, label_col=label))

table1 = pd.concat(table1_parts, ignore_index=True, sort=False)
save_table(table1, "table1_demographics_by_worldcup_year")
table1.head(25)


Saved table: D:\ai-job-agent\MIMIC_Project_3\outputs\neiss_world_cup_full\tables\table1_demographics_by_worldcup_year.csv



[executed in 212.5s]


In [ ]:

top_diagnosis = weighted_table_by_group(df, PRIMARY_WC_FLAG, "Diagnosis", label_col="Diagnosis_label", top_n=10)
top_bodypart = weighted_table_by_group(df, PRIMARY_WC_FLAG, "Body_Part", label_col="Body_Part_label", top_n=10)
top_location = weighted_table_by_group(df, PRIMARY_WC_FLAG, "Location", label_col="Location_label", top_n=10)
table2 = pd.concat([top_diagnosis, top_bodypart, top_location], ignore_index=True)
save_table(table2, "table2_top_injury_diagnosis_bodypart_location")
table2.head(30)


Saved table: D:\ai-job-agent\MIMIC_Project_3\outputs\neiss_world_cup_full\tables\table2_top_injury_diagnosis_bodypart_location.csv



[executed in 278.1s]


In [ ]:

weekly = (
    df.dropna(subset=["week_start"])
    .groupby("week_start")
    .agg(
        records=("case_id", "size"),
        weighted_estimate=("Weight", "sum"),
        soccer_records=("soccer_related", "sum"),
        soccer_weighted_estimate=("Weight", lambda x: x[df.loc[x.index, "soccer_related"]].sum()),
    )
    .reset_index()
)
weekly.to_csv(TABLE_DIR / "weekly_injury_counts.csv", index=False)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(weekly["week_start"], weekly["weighted_estimate"], color="#1f77b4", lw=1.8, label="All injuries, weighted")
if weekly["soccer_weighted_estimate"].sum() > 0:
    ax.plot(weekly["week_start"], weekly["soccer_weighted_estimate"], color="#d62728", lw=1.5, label="Soccer-related, weighted")

date_min, date_max = weekly["week_start"].min(), weekly["week_start"].max()
for _, ev in WORLD_CUP_EVENTS.iterrows():
    if pd.notna(date_min) and ev["end"] >= date_min and ev["start"] <= date_max:
        ax.axvspan(ev["start"], ev["end"], color="#f2c94c", alpha=0.25)
        ax.text(ev["start"], ax.get_ylim()[1] * 0.96, str(ev["year"]), fontsize=8, va="top", rotation=90)

ax.set_title("Figure 1. Weekly NEISS injury estimates with FIFA World Cup periods shaded")
ax.set_xlabel("Week start")
ax.set_ylabel("Weighted national estimate")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.25)
save_current_fig("figure1_weekly_counts_worldcup_shading")
weekly.tail()


Saved figure: D:\ai-job-agent\MIMIC_Project_3\outputs\neiss_world_cup_full\figures\figure1_weekly_counts_worldcup_shading.png



[executed in 11.0s]


In [ ]:

weekly_overlay = (
    df.dropna(subset=["week_of_year"])
    .groupby([PRIMARY_WC_FLAG, "week_of_year"])
    .agg(
        mean_weekly_weighted_estimate=("Weight", "sum"),
        records=("case_id", "size"),
        soccer_weighted_estimate=("Weight", lambda x: x[df.loc[x.index, "soccer_related"]].sum()),
    )
    .reset_index()
)
weekly_overlay.to_csv(TABLE_DIR / "weekly_overlay_worldcup_vs_nonworldcup.csv", index=False)

fig, ax = plt.subplots(figsize=(12, 5))
for group_value, g in weekly_overlay.groupby(PRIMARY_WC_FLAG):
    label = "World Cup years" if bool(group_value) else "Non-World Cup years"
    ax.plot(g["week_of_year"], g["mean_weekly_weighted_estimate"], marker="o", markersize=2.5, lw=1.5, label=label)
ax.set_title("Figure 2. Weekly injury estimates: World Cup versus non-World Cup years")
ax.set_xlabel("ISO week of year")
ax.set_ylabel("Weighted national estimate")
ax.legend()
ax.grid(True, alpha=0.25)
save_current_fig("figure2_overlay_weekly_counts")
weekly_overlay.head()


Saved figure: D:\ai-job-agent\MIMIC_Project_3\outputs\neiss_world_cup_full\figures\figure2_overlay_weekly_counts.png



[executed in 13.2s]


In [ ]:

heatmap_data = (
    df.dropna(subset=["month", "day_of_week"])
    .groupby([PRIMARY_WC_FLAG, "month", "day_of_week"])
    .agg(weighted_estimate=("Weight", "sum"), records=("case_id", "size"))
    .reset_index()
)
heatmap_data.to_csv(TABLE_DIR / "heatmap_day_month_data.csv", index=False)

month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
dow_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
groups = sorted(heatmap_data[PRIMARY_WC_FLAG].dropna().unique().tolist())
fig, axes = plt.subplots(1, len(groups), figsize=(6 * max(1, len(groups)), 5), squeeze=False)
for ax, group_value in zip(axes[0], groups):
    g = heatmap_data[heatmap_data[PRIMARY_WC_FLAG] == group_value]
    pivot = g.pivot(index="day_of_week", columns="month", values="weighted_estimate").reindex(index=range(7), columns=range(1, 13)).fillna(0)
    if sns:
        sns.heatmap(pivot, ax=ax, cmap="YlOrRd", cbar=True, xticklabels=month_labels, yticklabels=dow_labels)
    else:
        im = ax.imshow(pivot.values, aspect="auto", cmap="YlOrRd")
        ax.set_xticks(range(12), month_labels)
        ax.set_yticks(range(7), dow_labels)
        fig.colorbar(im, ax=ax)
    ax.set_title("World Cup years" if bool(group_value) else "Non-World Cup years")
    ax.set_xlabel("Month")
    ax.set_ylabel("Day of week")
save_current_fig("figure3_heatmap_day_month")
heatmap_data.head()


Saved figure: D:\ai-job-agent\MIMIC_Project_3\outputs\neiss_world_cup_full\figures\figure3_heatmap_day_month.png



[executed in 9.5s]


In [ ]:

KEYWORD_PATTERNS = {
    "soccer": r"\bsoccer\b",
    "football_contextual": r"\bfootball\b",
    "fifa_or_world_cup": r"\bfifa\b|\bworld cup\b",
    "tv_or_watching": r"\btv\b|\btelevision\b|\bwatch(?:ed|ing)?\b",
    "party_gathering": r"\bparty\b|\bgathering\b|\bbar\b|\bpub\b|\bcrowd\b",
    "celebration": r"\bcelebrat\w*\b|\bcheer\w*\b",
    "kick_or_kicked": r"\bkick(?:ed|ing)?\b",
    "goal_or_goalie": r"\bgoal\b|\bgoalie\b|\bgoalkeeper\b",
    "ball": r"\bball\b",
    "fall": r"\bfall\b|\bfell\b|\bfallen\b",
    "collision_hit": r"\bcollid\w*\b|\bhit\b|\bstruck\b",
    "alcohol": r"\balcohol\b|\bdrunk\b|\bintoxicat\w*\b",
}

keyword_rows = []
for name, pattern in KEYWORD_PATTERNS.items():
    flag = df["Narrative_clean"].str.contains(pattern, case=False, regex=True, na=False)
    for group_value, g in df.groupby(PRIMARY_WC_FLAG):
        idx = g.index
        keyword_rows.append({
            "keyword_theme": name,
            "group": group_value,
            "unweighted_n": int(flag.loc[idx].sum()),
            "weighted_estimate": df.loc[idx[flag.loc[idx]], "Weight"].sum() if flag.loc[idx].any() else 0.0,
            "weighted_pct_within_group": pct_fmt(df.loc[idx[flag.loc[idx]], "Weight"].sum() if flag.loc[idx].any() else 0.0, g["Weight"].sum()),
        })

context_summary = weighted_table_by_group(df, PRIMARY_WC_FLAG, "soccer_context_category", label_col="soccer_context_category")
table3 = pd.concat([pd.DataFrame(keyword_rows), context_summary.rename(columns={"level": "keyword_theme"})], ignore_index=True, sort=False)
save_table(table3, "table3_narrative_keywords_and_contexts")
table3.head(40)


Saved table: D:\ai-job-agent\MIMIC_Project_3\outputs\neiss_world_cup_full\tables\table3_narrative_keywords_and_contexts.csv



[executed in 602.5s]


In [ ]:
# Figure 4: Bar plot of soccer-related injury contexts and keywords.
plot_keyword = pd.DataFrame(keyword_rows)
top_kw = plot_keyword.groupby("keyword_theme")["weighted_estimate"].sum().sort_values(ascending=False).head(12).index
plot_keyword = plot_keyword[plot_keyword["keyword_theme"].isin(top_kw)].copy()
plot_keyword["group_label"] = np.where(plot_keyword["group"].astype(bool), "World Cup years", "Non-World Cup years")

fig, ax = plt.subplots(figsize=(10, 6))
if sns:
    sns.barplot(data=plot_keyword, x="weighted_estimate", y="keyword_theme", hue="group_label", ax=ax)
else:
    pivot = plot_keyword.pivot(index="keyword_theme", columns="group_label", values="weighted_estimate").fillna(0)
    pivot.plot(kind="barh", ax=ax)
ax.set_title("Figure 4. Narrative keyword themes in NEISS injury narratives")
ax.set_xlabel("Weighted national estimate")
ax.set_ylabel("Keyword/theme")
save_current_fig("figure4_narrative_keyword_barplot")
plot_keyword.head()


Saved figure: D:\ai-job-agent\MIMIC_Project_3\outputs\neiss_world_cup_full\figures\figure4_narrative_keyword_barplot.png



[executed in 0.8s]


In [ ]:
# Topic modeling on soccer-related narratives using LDA.
topic_source = df.loc[df["soccer_related"] & df["Narrative_clean"].str.len().gt(10), ["Narrative_clean", PRIMARY_WC_FLAG, "Weight"]].copy()
if len(topic_source) > 8000:
    topic_source = topic_source.sample(8000, random_state=RANDOM_STATE)

topic_rows = []
topic_doc_assignments = pd.DataFrame()
if len(topic_source) >= 50:
    vectorizer = CountVectorizer(stop_words="english", max_features=2500, min_df=5, ngram_range=(1, 2))
    X_counts = vectorizer.fit_transform(topic_source["Narrative_clean"])
    n_topics = min(6, max(2, int(len(topic_source) / 1000) + 2))
    lda = LatentDirichletAllocation(n_components=n_topics, random_state=RANDOM_STATE, learning_method="batch", max_iter=10)
    topic_matrix = lda.fit_transform(X_counts)
    terms = np.array(vectorizer.get_feature_names_out())
    for topic_idx, comp in enumerate(lda.components_):
        top_terms = terms[np.argsort(comp)[::-1][:12]]
        topic_rows.append({"topic_id": topic_idx, "top_terms": ", ".join(top_terms)})
    topic_doc_assignments = topic_source.copy()
    topic_doc_assignments["topic_id"] = topic_matrix.argmax(axis=1)
    topic_doc_assignments["topic_confidence"] = topic_matrix.max(axis=1)
else:
    topic_rows.append({"topic_id": "not_estimated", "top_terms": "Fewer than 50 soccer-related narratives available."})

topic_table = pd.DataFrame(topic_rows)
topic_summary = (
    topic_doc_assignments.groupby([PRIMARY_WC_FLAG, "topic_id"])
    .agg(records=("Narrative_clean", "size"), weighted_estimate=("Weight", "sum"), mean_confidence=("topic_confidence", "mean"))
    .reset_index()
    if len(topic_doc_assignments) else pd.DataFrame()
)
topic_table.to_csv(TABLE_DIR / "narrative_lda_topic_terms.csv", index=False)
topic_summary.to_csv(TABLE_DIR / "narrative_lda_topic_summary.csv", index=False)
topic_table



[executed in 28.2s]


In [ ]:
def prepare_model_data(data, outcome_col="soccer_related", max_rows=300000):
    model_df = data.dropna(subset=["Age_years", "Treatment_Date"]).copy()
    model_df = model_df[model_df["Sex"].isin([1, 2])].copy()
    model_df[outcome_col] = model_df[outcome_col].astype(int)
    needed = [outcome_col, PRIMARY_WC_FLAG, "Age_years", "Sex_label", "day_of_week_name", "season", "Weight"]
    model_df = model_df[needed].dropna().copy()
    if len(model_df) > max_rows:
        positives = model_df[model_df[outcome_col] == 1]
        negatives = model_df[model_df[outcome_col] == 0]
        n_neg = max_rows - len(positives)
        if n_neg > 0 and len(negatives) > n_neg:
            negatives = negatives.sample(n_neg, random_state=RANDOM_STATE)
        model_df = pd.concat([positives, negatives], ignore_index=True).sample(frac=1, random_state=RANDOM_STATE)
    return model_df

def fit_weighted_logistic_with_ci(data, outcome_col="soccer_related"):
    model_df = prepare_model_data(data, outcome_col=outcome_col)
    status = {
        "model": "Logistic: soccer_related ~ WorldCup_year + age + sex + day_of_week + season",
        "status": "not_fit",
        "reason": "",
        "n_records": len(model_df),
    }
    if model_df[outcome_col].nunique() < 2:
        status["reason"] = "Outcome has no variation in loaded data."
        return pd.DataFrame([status]), None, None
    if model_df[PRIMARY_WC_FLAG].nunique() < 2:
        status["reason"] = "World Cup year flag has no variation in loaded data. Add annual NEISS files from both World Cup and non-World Cup years."
        return pd.DataFrame([status]), None, None

    X = model_df[[PRIMARY_WC_FLAG, "Age_years", "Sex_label", "day_of_week_name", "season"]].copy()
    X[PRIMARY_WC_FLAG] = X[PRIMARY_WC_FLAG].astype(int)
    y = model_df[outcome_col].astype(int)
    w = model_df["Weight"].astype(float)

    categorical = ["Sex_label", "day_of_week_name", "season"]
    numeric = [PRIMARY_WC_FLAG, "Age_years"]
    pre = ColumnTransformer([
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), categorical),
        ("num", "passthrough", numeric),
    ])
    X_design = pre.fit_transform(X)
    feature_names = list(pre.get_feature_names_out())

    try:
        clf = LogisticRegression(penalty=None, solver="lbfgs", max_iter=1000, random_state=RANDOM_STATE)
    except TypeError:
        clf = LogisticRegression(C=1e6, solver="lbfgs", max_iter=1000, random_state=RANDOM_STATE)
    clf.fit(X_design, y, sample_weight=w)
    coef = clf.coef_.ravel()
    intercept = clf.intercept_[0]

    X_dense = np.asarray(X_design, dtype=float)
    X_aug = np.column_stack([np.ones(X_dense.shape[0]), X_dense])
    p_hat = expit(intercept + X_dense @ coef)
    W = np.clip(w.values * p_hat * (1 - p_hat), 1e-9, None)
    hessian = X_aug.T @ (X_aug * W[:, None])
    ridge = np.eye(hessian.shape[0]) * 1e-6
    try:
        cov = np.linalg.pinv(hessian + ridge)
        se = np.sqrt(np.diag(cov))
    except Exception:
        se = np.full(X_aug.shape[1], np.nan)

    all_names = ["Intercept"] + feature_names
    all_coef = np.r_[intercept, coef]
    z = all_coef / se
    p_values = 2 * (1 - norm.cdf(np.abs(z)))
    result = pd.DataFrame({
        "model": status["model"],
        "term": all_names,
        "coef_log_odds": all_coef,
        "std_error": se,
        "odds_ratio": np.exp(all_coef),
        "ci_lower": np.exp(all_coef - 1.96 * se),
        "ci_upper": np.exp(all_coef + 1.96 * se),
        "p_value": p_values,
        "n_records": len(model_df),
        "status": "fit",
        "reason": "",
    })

    y_prob = clf.predict_proba(X_design)[:, 1]
    metrics = pd.DataFrame([{
        "model": status["model"],
        "n_records": len(model_df),
        "auroc": roc_auc_score(y, y_prob),
        "average_precision": average_precision_score(y, y_prob),
        "event_rate": y.mean(),
    }])
    return result, metrics, (clf, pre, model_df)

table4_logistic, logistic_metrics, logistic_bundle = fit_weighted_logistic_with_ci(df)
save_table(table4_logistic, "table4_logistic_regression_soccer_related")
if logistic_metrics is not None:
    save_table(logistic_metrics, "table4_logistic_model_metrics")
table4_logistic.head(20)


Saved table: D:\ai-job-agent\MIMIC_Project_3\outputs\neiss_world_cup_full\tables\table4_logistic_regression_soccer_related.csv
Saved table: D:\ai-job-agent\MIMIC_Project_3\outputs\neiss_world_cup_full\tables\table4_logistic_model_metrics.csv



[executed in 29.0s]


In [ ]:
def fit_weekly_poisson(data):
    weekly_model = (
        data.dropna(subset=["week_start"])
        .groupby(["week_start", PRIMARY_WC_FLAG, "WorldCup_period_any", "same_calendar_wc_window", "season"])
        .agg(
            soccer_weighted_count=("Weight", lambda x: x[data.loc[x.index, "soccer_related"]].sum()),
            all_weighted_count=("Weight", "sum"),
            records=("case_id", "size"),
        )
        .reset_index()
    )
    weekly_model["week_index"] = (weekly_model["week_start"] - weekly_model["week_start"].min()).dt.days / 7
    status = {
        "model": "Poisson weekly soccer-related weighted count",
        "status": "not_fit",
        "reason": "",
        "n_weeks": len(weekly_model),
    }
    if len(weekly_model) < 10:
        status["reason"] = "Fewer than 10 weekly observations."
        return pd.DataFrame([status]), weekly_model, None
    if weekly_model[PRIMARY_WC_FLAG].nunique() < 2:
        status["reason"] = "World Cup year flag has no variation in loaded data."
        return pd.DataFrame([status]), weekly_model, None

    y = weekly_model["soccer_weighted_count"].astype(float).clip(lower=0).to_numpy()
    offset = np.log(weekly_model["all_weighted_count"].astype(float).clip(lower=1).to_numpy())
    X = pd.DataFrame({
        "Intercept": 1.0,
        "WorldCup_year_any": weekly_model[PRIMARY_WC_FLAG].astype(int).to_numpy(),
        "same_calendar_wc_window": weekly_model["same_calendar_wc_window"].astype(int).to_numpy(),
        "week_index_per_year": ((weekly_model["week_index"] - weekly_model["week_index"].mean()) / 52.0).to_numpy(),
    })
    season_dummies = pd.get_dummies(weekly_model["season"], prefix="season", drop_first=True, dtype=float)
    X = pd.concat([X, season_dummies.reset_index(drop=True)], axis=1)
    X_design = X.to_numpy(dtype=float)
    feature_names = X.columns.tolist()

    start_rate = max(y.sum() / weekly_model["all_weighted_count"].sum(), 1e-9)
    beta0 = np.zeros(X_design.shape[1])
    beta0[0] = np.log(start_rate)

    def nll(beta):
        eta = np.clip(offset + X_design @ beta, -30, 30)
        mu = np.exp(eta)
        return np.sum(mu - y * eta)

    def grad(beta):
        eta = np.clip(offset + X_design @ beta, -30, 30)
        mu = np.exp(eta)
        return X_design.T @ (mu - y)

    opt = minimize(nll, beta0, jac=grad, method="L-BFGS-B", options={"maxiter": 5000, "gtol": 1e-7})
    beta = opt.x
    eta = np.clip(offset + X_design @ beta, -30, 30)
    mu = np.exp(eta)
    weekly_model["predicted_soccer_weighted_count"] = mu

    hessian = X_design.T @ (X_design * mu[:, None])
    try:
        cov = np.linalg.pinv(hessian)
        se = np.sqrt(np.diag(cov))
    except Exception:
        se = np.full_like(beta, np.nan)
    p_values = 2 * (1 - norm.cdf(np.abs(beta / se)))

    result = pd.DataFrame({
        "model": status["model"] + " with log(total injuries) offset",
        "term": feature_names,
        "coef_log_rate": beta,
        "std_error": se,
        "rate_ratio": np.exp(beta),
        "ci_lower": np.exp(beta - 1.96 * se),
        "ci_upper": np.exp(beta + 1.96 * se),
        "p_value": p_values,
        "n_weeks": len(weekly_model),
        "converged": bool(opt.success),
        "status": "fit",
        "reason": "" if opt.success else str(opt.message),
    })
    return result, weekly_model, {"coef": beta, "features": feature_names, "converged": bool(opt.success)}

table4_poisson, weekly_model_data, poisson_bundle = fit_weekly_poisson(df)
save_table(table4_poisson, "table4_weekly_poisson_model")
weekly_model_data.to_csv(TABLE_DIR / "weekly_poisson_model_input_predictions.csv", index=False)
table4_poisson.head()


Saved table: D:\ai-job-agent\MIMIC_Project_3\outputs\neiss_world_cup_full\tables\table4_weekly_poisson_model.csv



[executed in 13.1s]


In [ ]:

fig, ax = plt.subplots(figsize=(9, 5))
if poisson_bundle is not None and "predicted_soccer_weighted_count" in weekly_model_data.columns:
    pred_plot = weekly_model_data.groupby(PRIMARY_WC_FLAG).agg(
        observed=("soccer_weighted_count", "mean"),
        predicted=("predicted_soccer_weighted_count", "mean"),
    ).reset_index()
    x = np.arange(len(pred_plot))
    ax.bar(x - 0.18, pred_plot["observed"], width=0.36, label="Observed weekly mean")
    ax.bar(x + 0.18, pred_plot["predicted"], width=0.36, label="Predicted weekly mean")
    ax.set_xticks(x, ["World Cup years" if bool(v) else "Non-World Cup years" for v in pred_plot[PRIMARY_WC_FLAG]])
    ax.set_ylabel("Soccer-related weighted weekly estimate")
    ax.set_title("Figure 5. Observed and predicted soccer-related weekly injury counts")
    ax.legend()
else:
    reason = table4_poisson.get("reason", pd.Series(["Model not available."])).iloc[0]
    ax.axis("off")
    ax.text(0.02, 0.75, "Figure 5 status", fontsize=14, weight="bold", transform=ax.transAxes)
    ax.text(0.02, 0.55, "Predicted counts require NEISS files from both World Cup and non-World Cup years.", fontsize=11, transform=ax.transAxes)
    ax.text(0.02, 0.40, f"Current status: {reason}", fontsize=10, transform=ax.transAxes)
save_current_fig("figure5_predicted_counts_or_model_status")


Saved figure: D:\ai-job-agent\MIMIC_Project_3\outputs\neiss_world_cup_full\figures\figure5_predicted_counts_or_model_status.png



[executed in 0.4s]


In [ ]:

embed_source = df.loc[df["Narrative_clean"].str.len().gt(10), [
    "Narrative_clean", "soccer_related", "soccer_context_category", "Age_years",
    "Body_Part_label", "Diagnosis_label", "Weight", PRIMARY_WC_FLAG
]].copy()

if len(embed_source) > 1000:
    soccer_part = embed_source[embed_source["soccer_related"]]
    other_part = embed_source[~embed_source["soccer_related"]]
    n_other = max(0, 1000 - min(len(soccer_part), 500))
    embed_source = pd.concat([
        soccer_part.sample(min(len(soccer_part), 500), random_state=RANDOM_STATE),
        other_part.sample(min(len(other_part), n_other), random_state=RANDOM_STATE),
    ]).sample(frac=1, random_state=RANDOM_STATE)

embedding_table = pd.DataFrame()
if len(embed_source) >= 100:
    tfidf = TfidfVectorizer(stop_words="english", max_features=5000, min_df=3, ngram_range=(1, 2))
    X_tfidf = tfidf.fit_transform(embed_source["Narrative_clean"])
    n_components = min(50, X_tfidf.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_components, random_state=RANDOM_STATE)
    X_svd = svd.fit_transform(X_tfidf)
    n_clusters = min(6, max(2, len(embed_source) // 400))
    km = KMeans(n_clusters=n_clusters, random_state=RANDOM_STATE, n_init="auto")
    clusters = km.fit_predict(X_svd)
    perplexity = min(30, max(5, (len(embed_source) - 1) // 3))
    tsne = TSNE(n_components=2, perplexity=perplexity, random_state=RANDOM_STATE, init="pca", learning_rate="auto", max_iter=500)
    coords = tsne.fit_transform(X_svd)
    embedding_table = embed_source.reset_index(drop=True).copy()
    embedding_table["cluster"] = clusters
    embedding_table["tsne_1"] = coords[:, 0]
    embedding_table["tsne_2"] = coords[:, 1]
    embedding_table.to_csv(TABLE_DIR / "figure6_embedding_sample_with_clusters.csv", index=False)

    cluster_summary = embedding_table.groupby("cluster").agg(
        records=("Narrative_clean", "size"),
        soccer_related_pct=("soccer_related", "mean"),
        mean_age=("Age_years", "mean"),
        weighted_estimate=("Weight", "sum"),
    ).reset_index()
    save_table(cluster_summary, "embedding_cluster_summary")

    fig, ax = plt.subplots(figsize=(8, 6))
    scatter = ax.scatter(
        embedding_table["tsne_1"],
        embedding_table["tsne_2"],
        c=embedding_table["cluster"],
        s=np.where(embedding_table["soccer_related"], 24, 8),
        alpha=0.7,
        cmap="tab10",
    )
    ax.set_title("Figure 6. Narrative and structured-feature clusters (t-SNE)")
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    ax.legend(*scatter.legend_elements(), title="Cluster", loc="best", fontsize=8)
else:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.axis("off")
    ax.text(0.02, 0.6, "Not enough narrative records for embedding analysis.", transform=ax.transAxes)
save_current_fig("figure6_narrative_embedding_tsne_clusters")
embedding_table.head()


Saved table: D:\ai-job-agent\MIMIC_Project_3\outputs\neiss_world_cup_full\tables\embedding_cluster_summary.csv
Saved figure: D:\ai-job-agent\MIMIC_Project_3\outputs\neiss_world_cup_full\figures\figure6_narrative_embedding_tsne_clusters.png



[executed in 12.5s]


In [ ]:

geospatial_audit = pd.DataFrame([
    {
        "field": "Location",
        "available": "Location" in df.columns,
        "interpretation": "NEISS injury location category, not state or geographic coordinates.",
        "usable_for_mapping": False,
    },
    {
        "field": "State/region",
        "available": any(c.lower() in ["state", "region", "census_region"] for c in df.columns),
        "interpretation": "Not present in the public annual workbook schema loaded here.",
        "usable_for_mapping": any(c.lower() in ["state", "region", "census_region"] for c in df.columns),
    },
])
geospatial_audit.to_csv(TABLE_DIR / "geospatial_availability_audit.csv", index=False)

location_by_wc = weighted_table_by_group(df, PRIMARY_WC_FLAG, "Location", label_col="Location_label", top_n=10)
save_table(location_by_wc, "location_patterns_by_worldcup_year")
geospatial_audit


Saved table: D:\ai-job-agent\MIMIC_Project_3\outputs\neiss_world_cup_full\tables\location_patterns_by_worldcup_year.csv



[executed in 100.7s]


In [ ]:

sensitivity_defs = {
    "combined_product_or_keyword": df["soccer_related"],
    "product_code_only": df["soccer_product_flag"],
    "narrative_keyword_only": df["soccer_keyword_flag"],
    "actual_worldcup_period_plus_14_days": df["WorldCup_period_any_plus14"],
}

sens_rows = []
for name, flag in sensitivity_defs.items():
    flag = flag.fillna(False).astype(bool)
    sens_rows.append({
        "analysis": name,
        "records": int(flag.sum()),
        "weighted_estimate": df.loc[flag, "Weight"].sum(),
        "weighted_pct_total": pct_fmt(df.loc[flag, "Weight"].sum(), df["Weight"].sum()),
    })
    for group_var in [PRIMARY_WC_FLAG, "age_group", "Location_label"]:
        for level, idx in df.groupby(group_var).groups.items():
            idx = list(idx)
            denom = df.loc[idx, "Weight"].sum()
            numer = df.loc[np.intersect1d(np.array(idx), flag[flag].index.values), "Weight"].sum()
            sens_rows.append({
                "analysis": f"{name} within {group_var}",
                "stratum": level,
                "records": int(flag.loc[idx].sum()),
                "weighted_estimate": numer,
                "weighted_pct_total": pct_fmt(numer, denom),
            })

sensitivity_table = pd.DataFrame(sens_rows)
save_table(sensitivity_table, "sensitivity_analyses_soccer_definitions_age_location")
sensitivity_table.head(30)


Saved table: D:\ai-job-agent\MIMIC_Project_3\outputs\neiss_world_cup_full\tables\sensitivity_analyses_soccer_definitions_age_location.csv



[executed in 177.9s]


In [ ]:

crosswalk = pd.DataFrame([
    {"requirement": "Clean and standardize NEISS structured variables", "status": "implemented", "output": "cleaning_audit.csv; load_log.csv"},
    {"requirement": "Create month/week/day/holiday/WorldCup flags", "status": "implemented", "output": "time_variable_audit_by_year.csv"},
    {"requirement": "Weighted NEISS estimates", "status": "implemented", "output": "all descriptive tables include weighted_estimate"},
    {"requirement": "Table 1 demographics", "status": "implemented", "output": "table1_demographics_by_worldcup_year.csv/xlsx"},
    {"requirement": "Table 2 top injury types/body parts", "status": "implemented", "output": "table2_top_injury_diagnosis_bodypart_location.csv/xlsx"},
    {"requirement": "Figure 1 weekly counts with shaded WC periods", "status": "implemented", "output": "figure1_weekly_counts_worldcup_shading.png"},
    {"requirement": "Figure 2 weekly overlay", "status": "implemented", "output": "figure2_overlay_weekly_counts.png"},
    {"requirement": "Figure 3 day/month heatmap", "status": "implemented", "output": "figure3_heatmap_day_month.png"},
    {"requirement": "Table 3 narrative keywords/themes", "status": "implemented", "output": "table3_narrative_keywords_and_contexts.csv/xlsx; narrative_lda_topic_terms.csv"},
    {"requirement": "Figure 4 word cloud/bar plot", "status": "implemented as bar plot", "output": "figure4_narrative_keyword_barplot.png"},
    {"requirement": "Table 4 regression coefficients", "status": "implemented; status row if non-estimable", "output": "table4_logistic_regression_soccer_related.csv/xlsx; table4_weekly_poisson_model.csv/xlsx"},
    {"requirement": "Figure 5 predicted injury counts", "status": "implemented; status panel if non-estimable", "output": "figure5_predicted_counts_or_model_status.png"},
    {"requirement": "Figure 6 embedding clusters", "status": "implemented", "output": "figure6_narrative_embedding_tsne_clusters.png; embedding_cluster_summary.csv/xlsx"},
    {"requirement": "Geospatial patterns if available", "status": "audited; public files contain injury location category, not state", "output": "geospatial_availability_audit.csv; location_patterns_by_worldcup_year.csv/xlsx"},
    {"requirement": "Sensitivity analyses", "status": "implemented", "output": "sensitivity_analyses_soccer_definitions_age_location.csv/xlsx"},
])
crosswalk.to_csv(TABLE_DIR / "analysis_requirement_crosswalk.csv", index=False)

manifest_rows = []
for subdir in [TABLE_DIR, FIGURE_DIR, MODEL_DIR]:
    for p in sorted(subdir.glob("*")):
        if p.is_file():
            manifest_rows.append({
                "artifact": p.name,
                "path": str(p),
                "type": p.suffix.lower().lstrip("."),
                "size_kb": round(p.stat().st_size / 1024, 1),
            })
manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(OUTPUT_DIR / "output_manifest.csv", index=False)

print("Output manifest:")
print(manifest.to_string(index=False, max_rows=200))
print("\nRequirement crosswalk:")
print(crosswalk.to_string(index=False))


Output manifest:
                                                 artifact                                                                                                                          path type  size_kb
                       analysis_requirement_crosswalk.csv                        D:\ai-job-agent\MIMIC_Project_3\outputs\neiss_world_cup_full\tables\analysis_requirement_crosswalk.csv  csv      1.6
                                       cleaning_audit.csv                                        D:\ai-job-agent\MIMIC_Project_3\outputs\neiss_world_cup_full\tables\cleaning_audit.csv  csv      0.1
                              data_availability_audit.csv                               D:\ai-job-agent\MIMIC_Project_3\outputs\neiss_world_cup_full\tables\data_availability_audit.csv  csv      2.3
                            embedding_cluster_summary.csv                             D:\ai-job-agent\MIMIC_Project_3\outputs\neiss_world_cup_full\tables\embedding_cluster_summary.csv  csv   


[executed in 0.0s]
